# Structure and calibration checks

Use this notebook when a scan looks wrong. It isolates the geometry stages from model reconstruction and makes the assumptions visible.

In [ ]:
from pathlib import Path
import sys
import matplotlib
_interactive = False
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
    _interactive = 'agg' not in str(matplotlib.get_backend()).lower()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
def show_plot():
    if _interactive:
        plt.show()
    plt.close()

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pyproject.toml').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from inside the FINAL ISIS repository')
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]
film_path = ROOT / 'data/raw/csa_verified_ksh_1972322002235.png'
profile_path = ROOT / 'configs/film_calibration_profile.json'
assert film_path.is_file(), 'The verified CSA sample is missing from data/raw/'

from isis_research.image_io import load_image
from scripts.pipeline.extract_scan_structure import extract_structure, write_overlay
from scripts.pipeline.fit_frequency_axis import fit_from_profile, load_json
from scripts.pipeline.fit_height_axis import fit_from_profile as fit_height
from scripts.pipeline.standardize_film_only_512 import collapse_duplicate_fallback

image = load_image(film_path)
structure = extract_structure(image)
profile = load_json(profile_path)
frequency = collapse_duplicate_fallback(fit_from_profile([item['x'] for item in structure['vertical_markers']['candidates']], image.shape, profile))
height = fit_height(structure, profile, frequency)
print('structure:', structure['status'], structure['warnings'])
print('frequency:', frequency['status'], frequency.get('warnings', []))
print('height:', height['status'], height.get('warnings', []))

## What to inspect

The overlay should show boundaries around the exposed ionogram, marker candidates on the vertical printed lines, and a plausible ruling lattice. A low marker count, irregular lattice, or edge-touching film region should remain a review signal rather than being silently repaired.

In [ ]:
plot_structure = dict(structure)
plot_structure['film_region'] = dict(structure['film_region'])
plot_path = ROOT / 'outputs/notebooks/02_calibration_overlay.png'
plot_path.parent.mkdir(parents=True, exist_ok=True)
write_overlay(plot_path, image, plot_structure, film_path.name)
plt.figure(figsize=(12, 7))
plt.imshow(plt.imread(plot_path))
plt.axis('off')
show_plot()

In [ ]:
print('frequency breakpoints')
for point in frequency.get('breakpoints', []):
    print(point)
print('height breakpoints')
for point in height.get('breakpoints', []):
    print(point)